In [2]:
import pandas as pd
import numpy as np
import joblib

In [3]:
# Load the modeling dataset
file_path = "../data/processed/behavior_change_dataset.csv"

df = pd.read_csv(file_path)

In [4]:
# Define time-based split boundaries

train_end = "2011-05"
validation_end = "2011-08"

# Training set
train_df = df[
    df["Month"] <= train_end
].copy()

# Validation set
validation_df = df[
    (df["Month"] > train_end) &
    (df["Month"] <= validation_end)
].copy()

# Test set
test_df = df[
    df["Month"] > validation_end
].copy()

print("Train period:",
      train_df["Month"].min(), "to", train_df["Month"].max())

print("Validation period:",
      validation_df["Month"].min(), "to", validation_df["Month"].max())

print("Test period:",
      test_df["Month"].min(), "to", test_df["Month"].max())

print("\nRows:")
print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

Train period: 2010-01 to 2011-05
Validation period: 2011-06 to 2011-08
Test period: 2011-09 to 2011-12

Rows:
Train: 12832
Validation: 2553
Test: 4266


In [5]:
behavior_aware_features = [
    "historical_active_months",
    "historical_transactions",
    "historical_spending",
    "previous_transaction_count",
    "previous_total_quantity",
    "previous_total_spending",
    "previous_average_transaction_value",
    "previous_unique_products",
    "months_since_previous"
]

In [6]:
# Get one real test sample for API validation

api_test_sample = test_df[behavior_aware_features].iloc[0]

api_test_sample

historical_active_months                 5.000000
historical_transactions                  6.000000
historical_spending                   3402.390000
previous_transaction_count               1.000000
previous_total_quantity                277.000000
previous_total_spending                584.910000
previous_average_transaction_value      26.586818
previous_unique_products                22.000000
months_since_previous                    2.000000
Name: 7, dtype: float64

In [7]:
# Prepare API request payload

api_payload = api_test_sample.to_dict()

api_payload

{'historical_active_months': 5.0,
 'historical_transactions': 6.0,
 'historical_spending': 3402.39,
 'previous_transaction_count': 1.0,
 'previous_total_quantity': 277.0,
 'previous_total_spending': 584.9100000000001,
 'previous_average_transaction_value': 26.586818181818185,
 'previous_unique_products': 22.0,
 'months_since_previous': 2.0}

In [8]:
# Load the same versioned model used by the API
api_model = joblib.load(
    "../models/xgboost/v1/model.joblib"
)

api_test_probability_notebook = float(
    api_model.predict_proba(
        api_test_sample.to_frame().T
    )[0][1]
)

api_test_prediction_notebook = int(
    api_test_probability_notebook >= 0.30
)

print("Notebook probability:", api_test_probability_notebook)
print("Notebook prediction:", api_test_prediction_notebook)

Notebook probability: 0.14841708540916443
Notebook prediction: 0
